In [0]:
-- Search kom_providers for a specific list of HCP names (case-insensitive, partial match)
WITH hcp AS (
  SELECT *
  FROM (
    VALUES
      ('Kevin','Kues'),
      ('Kara','Woolgar'),
      ('Frankie','Burney'),
      ('Ashley','Glenn'),
      ('Tim','Wood'),
      ('Jeff','Posadas'),
      ('Teresa','Hahn'),
      ('Diego', NULL),              -- single-name entry
      ('Theresa','Gonzales'),
      ('Kathryn','Gasperian'),
      ('Jennifer','Widmer'),
      ('Madeleine','Ciobanu'),
      ('Angela','Campbell'),
      ('Drea','Petersen'),
      ('Ari','Nouraee'),
      ('Katherine','Kim'),
      ('Olivia','Brown'),
      ('David','Stefanoni'),
      ('Lauren','Pfeil'),
      ('Colleen','Dansereau'),
      ('Lauren','O''Grady'),
      ('Shannon','Dixon'),
      ('Sharon','Anderson'),
      ('Alejandra','Gomez'),
      ('Lorien','King'),
      ('Cheryl','Clow'),
      ('Jillian','Soler'),
      ('Melissa','Byler'),
      ('Wing','Hung'),
      ('Jennifer','Baker'),
      ('Grace','Meier'),
      ('Todd','Snyder'),
      ('Monica','Borden'),
      ('Amanda','Prince'),
      ('Amy','Pesky'),
      ('Lauren','Wilson'),
      ('Erin','Kissenberg'),
      ('Paulo','Mendoza'),
      ('Kaleigh','Bright'),
      ('Sarah','Young'),
      ('Deeksha','Bali'),
      ('Kim','Stephens'),
      ('Jessica','Upshaw')
  ) AS v(first_name, last_name)
)
SELECT DISTINCT kp.*
FROM com_edp_prd.com_raw.kom_providers kp
JOIN hcp
  ON (
       kp.first_name ILIKE '%' || hcp.first_name || '%'
       AND (
             hcp.last_name IS NULL
             OR kp.last_name ILIKE '%' || hcp.last_name || '%'
           )
     );


In [0]:
WITH hcp(first_name, last_name) AS (
  VALUES
    ('Kevin','Kues'),
    ('Kara','Woolgar'),
    ('Frankie','Burney'),
    ('Ashley','Glenn'),
    ('Tim','Wood'),
    ('Jeff','Posadas'),
    ('Teresa','Hahn'),
    ('Diego', NULL),
    ('Theresa','Gonzales'),
    ('Kathryn','Gasperian'),
    ('Jennifer','Widmer'),
    ('Madeleine','Ciobanu'),
    ('Angela','Campbell'),
    ('Drea','Petersen'),
    ('Ari','Nouraee'),
    ('Katherine','Kim'),
    ('Olivia','Brown'),
    ('David','Stefanoni'),
    ('Lauren','Pfeil'),
    ('Colleen','Dansereau'),
    ('Lauren','O''Grady'),
    ('Shannon','Dixon'),
    ('Sharon','Anderson'),
    ('Alejandra','Gomez'),
    ('Lorien','King'),
    ('Cheryl','Clow'),
    ('Jillian','Soler'),
    ('Melissa','Byler'),
    ('Wing','Hung'),
    ('Jennifer','Baker'),
    ('Grace','Meier'),
    ('Todd','Snyder'),
    ('Monica','Borden'),
    ('Amanda','Prince'),
    ('Amy','Pesky'),
    ('Lauren','Wilson'),
    ('Erin','Kissenberg'),
    ('Paulo','Mendoza'),
    ('Kaleigh','Bright'),
    ('Sarah','Young'),
    ('Deeksha','Bali'),
    ('Kim','Stephens'),
    ('Jessica','Upshaw')
)
SELECT DISTINCT
  kp.*,
  org.npi            AS org_npi_from_org_row,
  org.organization_name AS org_name_from_org_row,
  COALESCE(org.organization_name, kp.organization_name) AS resolved_organization_name
FROM com_edp_prd.com_raw.kom_providers kp
LEFT JOIN com_edp_prd.com_raw.kom_providers org
  ON TRIM(kp.hco_primary_npi) = TRIM(org.npi)       -- org row identified by matching NPI
  AND org.provider_type ILIKE '%org%'               -- optional: prefer rows that look like organizations
JOIN hcp
  ON kp.first_name ILIKE '%' || hcp.first_name || '%'
 AND (hcp.last_name IS NULL OR kp.last_name ILIKE '%' || hcp.last_name || '%');


In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.reference_file_0109

In [0]:
CREATE OR REPLACE TEMP VIEW hco_org_details AS
WITH hco_npis AS (

  -- Medical: billing NPI
  SELECT DISTINCT
      BILLING_NPI AS HCO_NPI
  FROM com_edp_prd.com_raw.kom_medical_events
  WHERE BILLING_NPI IS NOT NULL
    AND SERVICE_DATE BETWEEN '2023-08-01' AND '2025-11-30'

  UNION

  -- Pharmacy: pharmacy NPI
  SELECT DISTINCT
      PHARMACY_NPI AS HCO_NPI
  FROM com_edp_prd.com_raw.kom_pharmacy_events
  WHERE PHARMACY_NPI IS NOT NULL
    AND TRANSACTION_RESULT = 'PAID'
    AND FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30'
)

SELECT DISTINCT
    n.HCO_NPI,
    p.ORGANIZATION_NAME,
    p.primary_specialty,
    p.secondary_specialty,
    p.PROVIDER_ADDRESS,
    p.PROVIDER_CITY,
    p.PROVIDER_STATE,
    p.PROVIDER_ZIP
FROM hco_npis n
JOIN com_edp_prd.com_raw.kom_providers p
  ON n.HCO_NPI = p.NPI
WHERE p.PROVIDER_TYPE = 'ORGANIZATION';


In [0]:
select * from hco_org_details

In [0]:
%sql
-- =====================================================================
-- Payer 360 | Supporting temp views (grouped / de-cluttered)
-- =====================================================================

-- ---------------------------------------------------------------------
-- 1) Elaprase treatment events (Medical + Pharmacy) used for patient/HCP metrics
-- ---------------------------------------------------------------------
CREATE OR REPLACE TEMP VIEW MPSII_TREATMENT_TABLE AS
SELECT *
FROM (
    -- Medical (Elaprase NDC)
    SELECT DISTINCT
        PATIENT_ID                                  AS PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI)      AS HCP_NPI,
        BILLING_NPI                                AS HCO_NPI,
        NDC11                                       AS CODE,
        MEDICAL_EVENT_ID                            AS EVENT_ID,
        SERVICE_DATE                                AS FILL_DATE,
        PLACE_OF_SERVICE                            AS PLACE_OF_SERVICE,
        KH_PLAN_ID                                  AS KH_PLAN,
        null                                        AS PHARMACY_CHANNEL,
        'MEDICAL_EVENTS'                            AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('54092070001','540920700')

    UNION

    -- Pharmacy (Elaprase NDC | Paid only)
    SELECT DISTINCT
        PATIENT_ID                                  AS PATIENT_ID,
        PRESCRIBER_NPI                              AS HCP_NPI,
        PHARMACY_NPI                                AS HCO_NPI,
        NDC11                                       AS CODE,
        PHARMACY_EVENT_ID                           AS EVENT_ID,
        FILL_DATE                                   AS FILL_DATE,
        NULL                                        AS PLACE_OF_SERVICE,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        PHARMACY_CHANNEL                            AS PHARMACY_CHANNEL,
        'PHARMACY_EVENTS'                           AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'

    UNION

    -- Medical (Elaprase Procedure codes)
    SELECT DISTINCT
        PATIENT_ID                                  AS PATIENT_ID,
        RENDERING_NPI                               AS HCP_NPI,
        BILLING_NPI                                AS HCO_NPI,
        PROCEDURE_CODE                              AS CODE,
        MEDICAL_EVENT_ID                            AS EVENT_ID,
        SERVICE_DATE                                AS FILL_DATE,
        PLACE_OF_SERVICE                            AS PLACE_OF_SERVICE,
        KH_PLAN_ID                                  AS KH_PLAN,
        null                                        AS PHARMACY_CHANNEL,
        'MEDICAL_EVENTS'                            AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN (
        '99601','99602','96365','96366','J1743','S9357','S9379',
        '38206','38230','38232','38240','38241','38242','38243','38250'
    )
) t
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30';

In [0]:
select count(distinct patient_id) from mpsii_treatment_table

In [0]:
%sql
-- Medical vs Pharmacy patient overlap
WITH p AS (
  SELECT DISTINCT
    PATIENT_ID,
    CASE
      WHEN TABLE_NAME = 'PHARMACY_EVENTS' THEN 'PHARMACY'
      WHEN TABLE_NAME = 'MEDICAL_EVENTS'  THEN 'MEDICAL'
      ELSE 'OTHER'
    END AS SOURCE
  FROM MPSII_TREATMENT_TABLE
  WHERE TABLE_NAME IN ('PHARMACY_EVENTS','MEDICAL_EVENTS')
),
flags AS (
  SELECT
    PATIENT_ID,
    MAX(CASE WHEN SOURCE = 'MEDICAL' THEN 1 ELSE 0 END)  AS has_medical,
    MAX(CASE WHEN SOURCE = 'PHARMACY' THEN 1 ELSE 0 END) AS has_pharmacy
  FROM p
  GROUP BY PATIENT_ID
)
SELECT
  CASE
    WHEN has_medical = 1 AND has_pharmacy = 1 THEN 'BOTH'
    WHEN has_medical = 1 AND has_pharmacy = 0 THEN 'MEDICAL_ONLY'
    WHEN has_medical = 0 AND has_pharmacy = 1 THEN 'PHARMACY_ONLY'
  END AS PATIENT_SOURCE_BUCKET,
  COUNT(*) AS DISTINCT_PATIENTS
FROM flags
GROUP BY 1
ORDER BY 1;


In [0]:
CREATE OR REPLACE TEMP VIEW MPSII_TREATMENT_TABLE AS
SELECT *
FROM (
    -- Medical (Elaprase NDC)
    SELECT DISTINCT
        PATIENT_ID                                  AS PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI)      AS HCP_NPI,
        BILLING_NPI                                AS HCO_NPI,
        NDC11                                       AS CODE,
        MEDICAL_EVENT_ID                            AS EVENT_ID,
        SERVICE_DATE                                AS FILL_DATE,
        PLACE_OF_SERVICE                            AS PLACE_OF_SERVICE,
        KH_PLAN_ID                                  AS KH_PLAN,
        null                                        AS PHARMACY_CHANNEL,
        'MEDICAL_EVENTS'                            AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('54092070001','540920700')

    UNION

    -- Pharmacy (Elaprase NDC | Paid only)
    SELECT DISTINCT
        PATIENT_ID                                  AS PATIENT_ID,
        PRESCRIBER_NPI                              AS HCP_NPI,
        PHARMACY_NPI                                AS HCO_NPI,
        NDC11                                       AS CODE,
        PHARMACY_EVENT_ID                           AS EVENT_ID,
        FILL_DATE                                   AS FILL_DATE,
        NULL                                        AS PLACE_OF_SERVICE,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        PHARMACY_CHANNEL                            AS PHARMACY_CHANNEL,
        'PHARMACY_EVENTS'                           AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'

    UNION

    -- Medical (Elaprase Procedure codes)
    SELECT DISTINCT
        PATIENT_ID                                  AS PATIENT_ID,
        RENDERING_NPI                               AS HCP_NPI,
        BILLING_NPI                                AS HCO_NPI,
        PROCEDURE_CODE                              AS CODE,
        MEDICAL_EVENT_ID                            AS EVENT_ID,
        SERVICE_DATE                                AS FILL_DATE,
        PLACE_OF_SERVICE                            AS PLACE_OF_SERVICE,
        KH_PLAN_ID                                  AS KH_PLAN,
        null                                        AS PHARMACY_CHANNEL,
        'MEDICAL_EVENTS'                            AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN (
        '99601','99602','96365','96366','J1743','S9357','S9379',
        '38206','38230','38232','38240','38241','38242','38243','38250'
    )
) t
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30';
select * from mpsii_treatment_table

In [0]:
CREATE OR REPLACE TEMP VIEW mpsii_treatment_with_territory AS
WITH prov AS (
  SELECT
    TRIM(CAST(NPI AS STRING)) AS npi_str,
    LPAD(SUBSTR(REGEXP_REPLACE(COALESCE(PROVIDER_ZIP,''), '[^0-9]', ''), 1, 5), 5, '0') AS zip5
  FROM com_edp_prd.com_raw.kom_providers
)
SELECT
  t.HCP_NPI AS HCP_NPI,
  t.* EXCEPT (HCP_NPI),
  z.territory_id,
  z.territory_name,
  z.region_id,
  z.region_name
FROM MPSII_TREATMENT_TABLE t
LEFT JOIN prov p
  ON TRIM(CAST(t.HCP_NPI AS STRING)) = p.npi_str
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
  ON p.zip5 = z.zipcode;
select * from mpsii_treatment_with_territory

In [0]:
SELECT *
FROM com_edp_prd.com_raw.kom_providers
WHERE LOWER(TRIM(FIRST_NAME)) = LOWER('Chester')
  AND LOWER(TRIM(LAST_NAME))  = LOWER('Whitley');

In [0]:
CREATE OR REPLACE TEMP VIEW runtime_parameters AS

SELECT
    (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events) AS max_medical_date,

    (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events) AS max_pharmacy_date,

    LAST_DAY(
        ADD_MONTHS(
            LEAST(
                (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events),
                (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events)
            ), -1
        )
    ) AS end_date,

    CURRENT_DATE() AS run_date;
    select * from runtime_parameters

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping